In [3]:
from pathlib import Path
import re
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
import sys


PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SPLITS = [
    "train",
    "dev",
    "test",
    "incoming",
]

In [2]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from bs4 import BeautifulSoup, Comment
from scipy.stats import bootstrap


In [ ]:
# Number of bootstrap replications.
# 10,000 is a good default for publication-quality results.
N_BOOT = 10_000
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42
TIME_IS_MILLISECONDS = True

#### HTML parsing

In [6]:
# ============================================================
# HTML parsing
# ============================================================

def extract_metadata(soup: BeautifulSoup) -> dict:
    """
    Extract the JSON metadata stored inside the HTMLLabelizer comment.

    Expected structure:

        <!-- HTMLLabelizer
        {
            ...
            "meta": {
                "time": 7691270,
                ...
            }
        }
        -->
    """

    for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
        comment_text = str(comment).strip()

        if not comment_text.startswith("HTMLLabelizer"):
            continue

        # Remove the "HTMLLabelizer" prefix.
        json_text = comment_text[len("HTMLLabelizer"):].strip()

        try:
            data = json.loads(json_text)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Could not parse HTMLLabelizer JSON: {e}"
            )

        if "meta" not in data:
            raise ValueError("HTMLLabelizer JSON has no 'meta' field.")

        return data["meta"]

    raise ValueError("Could not find HTMLLabelizer comment.")


def count_spans(soup: BeautifulSoup) -> int:
    """
    Count every manual_label and auto_label span.

    Both are treated identically.
    """

    return len(soup.find_all(["manual_label", "auto_label"]))


def count_words(soup: BeautifulSoup) -> int:
    """
    Count word tokens in the document text.

    We deliberately exclude HTML comments, scripts, styles, etc.

    The tokenizer treats sequences of Unicode letters/numbers as words
    and keeps internal apostrophes/hyphens inside a word.
    """

    # Work from <body> when available.
    if soup.body is not None:
        text = soup.body.get_text(" ", strip=True)
    else:
        text = soup.get_text(" ", strip=True)

    # Remove script/style text if any somehow survives.
    # The BeautifulSoup get_text() normally handles this poorly if
    # they are present, so remove them explicitly beforehand would
    # be preferable; this is just a defensive step.
    #
    # Word token:
    #   maison
    #   Québec
    #   court's
    #   decision-making
    #
    # Unicode-aware because Python's regex \w handles Unicode.
    tokens = re.findall(
        r"[^\W_]+(?:['’\-][^\W_]+)*",
        text,
        flags=re.UNICODE,
    )

    return len(tokens)


def parse_document(path: Path) -> dict:
    """
    Read one HTML annotation file and extract:

        - split
        - document path
        - annotation time
        - span count
        - word count
    """

    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    soup = BeautifulSoup(text, "html.parser")

    meta = extract_metadata(soup)

    if "time" not in meta:
        raise ValueError("Metadata contains no 'time' field.")

    time_raw = float(meta["time"])

    if TIME_IS_MILLISECONDS:
        time_hours = time_raw / 1000.0 / 3600.0
    else:
        time_hours = time_raw / 3600.0

    if time_hours <= 0:
        raise ValueError(f"Invalid annotation time: {time_raw}")

    spans = count_spans(soup)
    words = count_words(soup)

    return {
        "file": str(path),
        "time_hours": time_hours,
        "spans": spans,
        "words": words,
        "span_rate": spans / time_hours,
        "word_rate": words / time_hours,
    }




#### Load all documents

In [7]:
# ============================================================
# Load all documents
# ============================================================

def load_documents() -> List[dict]:

    documents = []

    for split in SPLITS:

        split_dir = PROJECT_ROOT / "data" / "annotated" / split

        if not split_dir.exists():
            print(f"[WARNING] Split does not exist: {split_dir}")
            continue

        files = sorted(split_dir.rglob("*.html"))

        if not files:
            print(f"[WARNING] No HTML files in: {split_dir}")
            continue

        print(f"\n--- {split}: {len(files)} HTML files ---")

        for f in files:

            try:
                doc = parse_document(f)

                doc["split"] = split

                documents.append(doc)

                print(
                    f"  {f.name:50s} "
                    f"time={doc['time_hours']:.3f} h, "
                    f"spans={doc['spans']:5d}, "
                    f"words={doc['words']:7d}"
                )

            except Exception as e:
                print(f"[ERROR] {f}: {e}")

    return documents



#### Productivity statistics

In [8]:
# ============================================================
# Productivity statistics
# ============================================================

def aggregate_rate(
    documents: List[dict],
    numerator: str,
) -> float:
    """
    Ratio-of-sums productivity estimator:

        total numerator / total annotation time
    """

    total = sum(d[numerator] for d in documents)
    total_hours = sum(d["time_hours"] for d in documents)

    return total / total_hours


def mean_document_rate(
    documents: List[dict],
    rate_key: str,
) -> float:
    """
    Simple mean of document-level rates.

    This is useful as a secondary descriptive statistic,
    but NOT our primary productivity estimator.
    """

    return np.mean([d[rate_key] for d in documents])

#### Bootstrap

In [9]:
# ============================================================
# Bootstrap
# ============================================================

def ratio_rate_from_arrays(
    numerator: np.ndarray,
    time_hours: np.ndarray,
    axis: int = -1,
) -> np.ndarray:
    """
    Ratio-of-sums statistic used by scipy.stats.bootstrap.
    """

    return np.sum(numerator, axis=axis) / np.sum(time_hours, axis=axis)


def bootstrap_ci_single_group(
    documents: List[dict],
    numerator: str,
    confidence_level: float = 0.95,
    n_boot: int = 10_000,
    seed: int = 42,
) -> Tuple[float, float, float]:
    """
    BCa bootstrap CI for a single group's ratio-of-sums rate.
    """

    numerator_values = np.asarray(
        [d[numerator] for d in documents],
        dtype=float,
    )

    time_values = np.asarray(
        [d["time_hours"] for d in documents],
        dtype=float,
    )

    estimate = ratio_rate_from_arrays(
        numerator_values,
        time_values,
    )

    def statistic(numerator, time_hours, axis=-1):
        return ratio_rate_from_arrays(
            numerator,
            time_hours,
            axis=axis,
        )

    result = bootstrap(
        data=(numerator_values, time_values),
        statistic=statistic,
        n_resamples=n_boot,
        confidence_level=confidence_level,
        method="BCa",
        vectorized=True,
        paired=True,
        random_state=np.random.default_rng(seed),
    )

    return (
        estimate,
        result.confidence_interval.low,
        result.confidence_interval.high,
    )


def bootstrap_ci_difference(
    group_a: List[dict],
    group_b: List[dict],
    numerator: str,
    confidence_level: float = 0.95,
    n_boot: int = 10_000,
    seed: int = 42,
) -> Tuple[float, float, float]:
    """
    BCa bootstrap CI for the difference between two
    ratio-of-sums productivity estimates.

    Difference is:

        rate_A - rate_B
    """

    a_num = np.asarray(
        [d[numerator] for d in group_a],
        dtype=float,
    )

    a_time = np.asarray(
        [d["time_hours"] for d in group_a],
        dtype=float,
    )

    b_num = np.asarray(
        [d[numerator] for d in group_b],
        dtype=float,
    )

    b_time = np.asarray(
        [d["time_hours"] for d in group_b],
        dtype=float,
    )

    observed = (
        np.sum(a_num) / np.sum(a_time)
        -
        np.sum(b_num) / np.sum(b_time)
    )

    def statistic(
        a_num,
        a_time,
        b_num,
        b_time,
        axis=-1,
    ):
        rate_a = (
            np.sum(a_num, axis=axis)
            /
            np.sum(a_time, axis=axis)
        )

        rate_b = (
            np.sum(b_num, axis=axis)
            /
            np.sum(b_time, axis=axis)
        )

        return rate_a - rate_b

    result = bootstrap(
        data=(a_num, a_time, b_num, b_time),
        statistic=statistic,
        n_resamples=n_boot,
        confidence_level=confidence_level,
        method="BCa",
        vectorized=True,
        paired=False,
        random_state=np.random.default_rng(seed),
    )

    return (
        observed,
        result.confidence_interval.low,
        result.confidence_interval.high,
    )

#### Reporting

In [10]:
# ============================================================
# Reporting
# ============================================================

def print_group_summary(
    name: str,
    documents: List[dict],
):

    total_time = sum(d["time_hours"] for d in documents)
    total_spans = sum(d["spans"] for d in documents)
    total_words = sum(d["words"] for d in documents)

    span_rate = total_spans / total_time
    word_rate = total_words / total_time

    mean_span_rate = mean_document_rate(
        documents,
        "span_rate",
    )

    mean_word_rate = mean_document_rate(
        documents,
        "word_rate",
    )

    print(f"\n{name}")
    print("-" * len(name))

    print(f"# documents       : {len(documents)}")
    print(f"Total annotation  : {total_time:.2f} hours")
    print(f"Total spans       : {total_spans:,}")
    print(f"Total words       : {total_words:,}")

    print()
    print("Ratio-of-sums productivity:")
    print(f"  spans/hour      : {span_rate:,.2f}")
    print(f"  words/hour      : {word_rate:,.2f}")

    print()
    print("Mean document-level productivity:")
    print(f"  spans/hour      : {mean_span_rate:,.2f}")
    print(f"  words/hour      : {mean_word_rate:,.2f}")

#### Main

In [11]:

# ============================================================
# Main
# ============================================================

def main():

    print("=" * 80)
    print("ANNOTATION TIME / PRODUCTIVITY ANALYSIS")
    print("=" * 80)

    documents = load_documents()

    if not documents:
        raise RuntimeError("No documents found.")

    # --------------------------------------------------------
    # Split documents into the two populations we want to
    # compare.
    #
    # Group A = train + dev + test
    # Group B = incoming
    # --------------------------------------------------------

    annotated = [
        d
        for d in documents
        if d["split"] in {"train", "dev", "test"}
    ]

    incoming = [
        d
        for d in documents
        if d["split"] == "incoming"
    ]

    if not annotated:
        raise RuntimeError("No train/dev/test documents found.")

    if not incoming:
        raise RuntimeError("No incoming documents found.")

    # --------------------------------------------------------
    # Descriptive statistics
    # --------------------------------------------------------

    print("\n\n")
    print("=" * 80)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 80)

    print_group_summary(
        "TRAIN + DEV + TEST",
        annotated,
    )

    print_group_summary(
        "INCOMING",
        incoming,
    )

    # --------------------------------------------------------
    # Bootstrap CIs for each group
    # --------------------------------------------------------

    print("\n\n")
    print("=" * 80)
    print("95% BCa BOOTSTRAP CONFIDENCE INTERVALS")
    print("=" * 80)

    for numerator, label in [
        ("spans", "Spans/hour"),
        ("words", "Words/hour"),
    ]:

        print(f"\n{label}")
        print("-" * len(label))

        for name, group in [
            ("train+dev+test", annotated),
            ("incoming", incoming),
        ]:

            estimate, low, high = bootstrap_ci_single_group(
                group,
                numerator=numerator,
                confidence_level=CONFIDENCE_LEVEL,
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )

            print(
                f"{name:20s}: "
                f"{estimate:,.2f} "
                f"[{low:,.2f}, {high:,.2f}]"
            )

    # --------------------------------------------------------
    # Bootstrap CI for difference
    # --------------------------------------------------------

    print("\n\n")
    print("=" * 80)
    print("DIFFERENCE BETWEEN GROUPS")
    print("=" * 80)

    for numerator, label in [
        ("spans", "Spans/hour"),
        ("words", "Words/hour"),
    ]:

        difference, low, high = bootstrap_ci_difference(
            annotated,
            incoming,
            numerator=numerator,
            confidence_level=CONFIDENCE_LEVEL,
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        )

        annotated_rate = aggregate_rate(
            annotated,
            numerator,
        )

        incoming_rate = aggregate_rate(
            incoming,
            numerator,
        )

        relative_difference = (
            (incoming_rate / annotated_rate) - 1
        ) * 100

        print(f"\n{label}")
        print("-" * len(label))

        print(
            f"Train/dev/test rate : "
            f"{annotated_rate:,.2f}"
        )

        print(
            f"Incoming rate       : "
            f"{incoming_rate:,.2f}"
        )

        print(
            f"Difference          : "
            f"{difference:,.2f}"
        )

        print(
            f"95% BCa CI          : "
            f"[{low:,.2f}, {high:,.2f}]"
        )

        print(
            f"Relative difference : "
            f"{relative_difference:+.2f}%"
        )

    # --------------------------------------------------------
    # Per-split summaries
    # --------------------------------------------------------

    print("\n\n")
    print("=" * 80)
    print("INDIVIDUAL SPLITS")
    print("=" * 80)

    for split in SPLITS:

        group = [
            d for d in documents
            if d["split"] == split
        ]

        if not group:
            continue

        total_time = sum(d["time_hours"] for d in group)

        print(
            f"\n{split:10s}: "
            f"{len(group):3d} docs | "
            f"{total_time:8.2f} h | "
            f"{aggregate_rate(group, 'spans'):8.2f} spans/h | "
            f"{aggregate_rate(group, 'words'):8.2f} words/h"
        )




In [12]:
main()

ANNOTATION TIME / PRODUCTIVITY ANALYSIS

--- train: 4 HTML files ---
  1994CanLII4528NLCA.html                            time=2.740 h, spans=  424, words=  10215
  1997CanLII16226ONCA.html                           time=9.929 h, spans= 1911, words=  40713
  2008CSC9.html                                      time=7.135 h, spans= 1399, words=  28101
  2019SCC65.html                                     time=23.853 h, spans= 4658, words=  62969

--- dev: 3 HTML files ---
  1989CanLII1415ONCA.html                            time=2.136 h, spans=  241, words=   4285
  2016NBOMB12.html                                   time=4.473 h, spans=  306, words=   7578
  2021QCCA1675.html                                  time=2.191 h, spans=  271, words=   4640

--- test: 4 HTML files ---
  2001CanLII21117.html                               time=4.738 h, spans=  789, words=  16929
  2002SCC33.html                                     time=7.175 h, spans=  953, words=  34903
  2005QCCA437.html           